# R1 — Access Control (Governance)

Row-level and column-level access control for the identifier(:catalog) lakehouse: adjuster teams get state-scoped access to `identifier(:catalog).gold.identifier(:catalog)_enriched` via a row filter, national ID / phone PII in `identifier(:catalog).raw.members` is masked behind a last-4 mask, `account users` receive narrowly scoped grants, and compliance gets proof tables plus a point-in-time grants snapshot in `identifier(:catalog).gov`.

> Written per spec, not executed. Cells must be run top-to-bottom in order.

In [0]:
-- R1.1 ----------------------------------------------------------------
-- Governance schema + the identifier(:catalog).raw.* base tables the R1 spec references.
-- identifier(:catalog).raw exists as the landing-volume schema but has no tables yet,
-- so members / plans / hospitals are materialized here from bronze.

create schema if not exists identifier(:catalog).gov;

create or replace table identifier(:catalog).raw.members as
select member_id, name, dob, gender, city, state, national_id, phone, email
from identifier(:catalog).bronze.members;

create or replace table identifier(:catalog).raw.plans as
select plan_code, plan_family, sum_insured, room_rent_cap_per_day, copay_pct,
       ped_waiting_months, initial_waiting_days, annual_premium
from identifier(:catalog).bronze.plans;

create or replace table identifier(:catalog).raw.hospitals as
select hospital_id, name, city, city_tier, state
from identifier(:catalog).bronze.hospitals;

In [0]:
-- R1.2 ----------------------------------------------------------------
-- Persona registries the mask and row-filter functions consult at query time.
--
--   admin_users : principals allowed to see full PII values.
--   user_states : adjuster -> state mapping (mapped to Karnataka, Telangana).
--
-- TODO: replace the placeholder adjuster principals with the REAL account
-- user names of the regional adjuster test users before the live proof.

create or replace table identifier(:catalog).gov.admin_users (
  user_name string
);

insert into identifier(:catalog).gov.admin_users values
  ('deepaksinghal@cyntexa.com');              -- compliance admin / notebook owner

create or replace table identifier(:catalog).gov.user_states (
  user_name string,
  state     string
);

insert into identifier(:catalog).gov.user_states values
  ('adjuster.ka@example.com', 'Karnataka'),   -- TODO real adjuster account user
  ('adjuster.tg@example.com', 'Telangana');   -- TODO real adjuster account user

In [0]:
-- R1.3 ----------------------------------------------------------------
-- PII column mask: compliance admins (identifier(:catalog).gov.admin_users) see the value
-- in full; everyone else sees exactly four asterisks followed by the last
-- 4 characters (e.g. ****1234). Attached to identifier(:catalog).raw.members.national_id
-- and identifier(:catalog).raw.members.phone.
-- Note: CURRENT_USER()-based logic always evaluates as the INVOKER, so the
-- mask is enforced per reader regardless of table ownership.

create or replace function identifier(:catalog).gov.mask_last4(v string)
returns string
comment 'Column mask: admin_users see the full value; others see ****<last 4>.'
return case
         when v is null then null
         when exists (select 1
                      from identifier(:catalog).gov.admin_users a
                      where lower(a.user_name) = lower(current_user()))
           then v
         else concat('****', right(v, 4))
       end;

alter table identifier(:catalog).raw.members alter column national_id set mask identifier(:catalog).gov.mask_last4;
alter table identifier(:catalog).raw.members alter column phone      set mask identifier(:catalog).gov.mask_last4;

In [0]:
-- R1.4 ----------------------------------------------------------------
-- State row filter on identifier(:catalog).gold.identifier(:catalog)_enriched, keyed on hospital_id:
--   * compliance admin                     -> sees every claim
--   * adjuster mapped to a set of states   -> only identifier(:catalog) from hospitals
--                                             located in those states
--   * someone mapped to nothing            -> sees nothing at all

create or replace function identifier(:catalog).gov.identifier(:catalog)_state_filter(p_hospital_id string)
returns boolean
comment 'Row filter: admins see all identifier(:catalog); adjusters only hospitals in their mapped states; unmapped users see nothing.'
return
  exists (select 1
          from identifier(:catalog).gov.admin_users a
          where lower(a.user_name) = lower(current_user()))
  or exists (
       select 1
       from identifier(:catalog).gov.user_states us
       join identifier(:catalog).raw.hospitals h
         on h.state = us.state
       where lower(us.user_name) = lower(current_user())
         and h.hospital_id = p_hospital_id
     );

-- identifier(:catalog)_enriched is a materialized view, so the filter is attached with
-- ALTER MATERIALIZED VIEW. CURRENT_USER() runs as the invoker at query time.
alter materialized view identifier(:catalog).gold.identifier(:catalog)_enriched
  set row filter identifier(:catalog).gov.identifier(:catalog)_state_filter on (hospital_id);

-- NOTE: identifier(:catalog)_enriched is owned by the identifier(:catalog)_pipeline SDP pipeline; a
-- `create or refresh` of the gold view can drop the applied row filter.
-- Re-run the ALTER above after any pipeline refresh.

In [0]:
-- R1.5 ----------------------------------------------------------------
-- Grants to the account-level adjuster group, exactly as scoped by the spec:
--   use on identifier(:catalog)
--   use + read on identifier(:catalog).gold
--   use on identifier(:catalog).raw
--   read on identifier(:catalog).raw.plans and identifier(:catalog).raw.hospitals ONLY
-- (members PII stays unreachable directly — it is only visible via the
--  mask, or through the proof tables below.)

grant use catalog on catalog identifier(:catalog) to `account users`;

grant use schema on schema identifier(:catalog).gold to `account users`;
grant select     on schema identifier(:catalog).gold to `account users`;

grant use schema on schema identifier(:catalog).raw to `account users`;
grant select     on table identifier(:catalog).raw.plans     to `account users`;
grant select     on table identifier(:catalog).raw.hospitals to `account users`;

In [0]:
-- R1.6 ----------------------------------------------------------------
-- Compliance proof: the same 10 members, what an ADMIN sees vs what a
-- NON-ADMIN sees. Columns per spec: member_id, national_id, phone (STRING).
--
--   proof_members_admin  : materialized by a principal listed in
--                          admin_users, so the attached mask returns the
--                          values in full — this is a live pass through
--                          the real mask, not a copy of the raw table.
--   proof_members_masked : what the mask yields for a non-admin. The
--                          notebook author is an admin, so the mask itself
--                          would return raw values in this session; the
--                          masked table applies the exact mask expression
--                          (****<last 4>). For the end-to-end live check,
--                          have a mapped adjuster query the members table
--                          directly with a temporary SELECT grant.

create or replace table identifier(:catalog).gov.proof_members_admin as
select cast(member_id   as string) as member_id,
       cast(national_id as string) as national_id,
       cast(phone       as string) as phone
from identifier(:catalog).raw.members
order by member_id
limit 10;

create or replace table identifier(:catalog).gov.proof_members_masked as
select cast(member_id as string)                            as member_id,
       cast(concat('****', right(national_id, 4)) as string) as national_id,
       cast(concat('****', right(phone, 4))       as string) as phone
from identifier(:catalog).raw.members
order by member_id
limit 10;

In [0]:
-- R1.7 ----------------------------------------------------------------
-- Point-in-time record of exactly who can read what across the identifier(:catalog)
-- catalog: every catalog-, schema-, table-, view- and MV-level privilege,
-- in one auditable table. Must be (re)run AFTER R1.5 so the new grants
-- are captured.
-- Columns per spec: securable, principal, action_type, object_type,
--                   object_key (all STRING; securable == object_key is
--                   the fully qualified object name).

create or replace table identifier(:catalog).gov.grants_snapshot as
select cast(securable   as string) as securable,
       cast(principal   as string) as principal,
       cast(action_type as string) as action_type,
       cast(object_type as string) as object_type,
       cast(object_key  as string) as object_key
from (
  -- catalog-level privileges
  select catalog_name   as securable,
         grantee        as principal,
         privilege_type as action_type,
         'CATALOG'      as object_type,
         catalog_name   as object_key
  from identifier(:catalog).information_schema.catalog_privileges

  union all

  -- schema-level privileges
  select concat(catalog_name, '.', schema_name) as securable,
         grantee                                as principal,
         privilege_type                         as action_type,
         'SCHEMA'                               as object_type,
         concat(catalog_name, '.', schema_name) as object_key
  from identifier(:catalog).information_schema.schema_privileges

  union all

  -- table / view / MV / streaming-table privileges
  select concat(tp.table_catalog, '.', tp.table_schema, '.', tp.table_name)   as securable,
         tp.grantee                                                           as principal,
         tp.privilege_type                                                    as action_type,
         case t.table_type
           when 'MANAGED'           then 'TABLE'
           when 'EXTERNAL'          then 'TABLE'
           when 'VIEW'              then 'VIEW'
           when 'MATERIALIZED_VIEW' then 'MATERIALIZED_VIEW'
           when 'STREAMING_TABLE'   then 'STREAMING_TABLE'
           else t.table_type
         end                                                                  as object_type,
         concat(tp.table_catalog, '.', tp.table_schema, '.', tp.table_name)   as object_key
  from identifier(:catalog).information_schema.table_privileges tp
  left join identifier(:catalog).information_schema.tables t
         on  t.table_catalog = tp.table_catalog
         and t.table_schema  = tp.table_schema
         and t.table_name    = tp.table_name
);

-- Review: the R1.5 grants should be visible here —
--   `account users` -> USE on identifier(:catalog), USE + SELECT on identifier(:catalog).gold,
--                      USE on identifier(:catalog).raw, SELECT on identifier(:catalog).raw.plans
--                      and identifier(:catalog).raw.hospitals only.
select *
from identifier(:catalog).gov.grants_snapshot
order by object_key, principal;